# 🖥️ Laptop Price Prediction
## Step 5: Data Preprocessing Pipeline
**Student:** Shivakumar | B.Tech CSE | VEMU Institute of Technology

---
## 📦 0. Install & Import Libraries

In [ ]:
!pip install scikit-learn joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler,
    LabelEncoder, OrdinalEncoder, OneHotEncoder
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.impute import SimpleImputer

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12})

print("✅ All libraries imported")


---
## 📂 1. Load Feature Engineered Dataset

In [ ]:
# Upload laptop_price_featured.csv before running
df = pd.read_csv('laptop_price_featured.csv')

print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns:")
for col in df.columns:
    print(f"  • {col:<25} dtype: {df[col].dtype}  nulls: {df[col].isnull().sum()}")
df.head()


---
## 🎯 2. Define Features & Target

In [ ]:
# Target: use log_price for training (better model performance)
# Price_euros kept for final evaluation only

TARGET      = 'log_price'
DROP_COLS   = ['Price_euros', 'log_price',
               'Ram', 'Weight', 'Cpu', 'Gpu',        # raw text cols replaced by parsed versions
               'Memory', 'ScreenResolution', 'OpSys'] # raw cols replaced by engineered versions

# Drop raw text cols that were replaced
drop_existing = [c for c in DROP_COLS if c in df.columns]
X = df.drop(columns=drop_existing)
y = df[TARGET]

print(f"Feature matrix X : {X.shape}")
print(f"Target vector  y : {y.shape}  (log_price)")
print(f"\nFeatures used ({len(X.columns)}):")
for col in X.columns:
    print(f"  • {col:<25} dtype: {X[col].dtype}")


---
## 🗂️ 3. Identify Column Types for Pipeline

In [ ]:
# Separate column types
numerical_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()
binary_cols = [c for c in numerical_cols if X[c].nunique() == 2]

# Remove binary from numerical (they don't need scaling)
numerical_cols = [c for c in numerical_cols if c not in binary_cols]

print("Numerical columns (will be scaled):")
for c in numerical_cols: print(f"  • {c}")

print(f"\nCategorical columns (will be encoded):")
for c in categorical_cols: print(f"  • {c}")

print(f"\nBinary columns (pass-through):")
for c in binary_cols: print(f"  • {c}")


---
## ✂️ 4. Train-Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"✅ Train-Test Split complete")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"  y_test  : {y_test.shape}")
print(f"  Train % : {len(X_train)/len(X)*100:.1f}%")
print(f"  Test  % : {len(X_test)/len(X)*100:.1f}%")

# Visualise split
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(['Train', 'Test'], [len(X_train), len(X_test)],
    color=['#2E5D9E', '#C55A11'], edgecolor='white', alpha=0.9)
ax.set_title('Train-Test Split', fontweight='bold')
ax.set_xlabel('Number of Samples')
for i, v in enumerate([len(X_train), len(X_test)]):
    ax.text(v + 5, i, str(v), va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('pp_01_split.png', bbox_inches='tight')
plt.show()

print("""
📌 Decision: 80/20 split with random_state=42 for reproducibility.
   Stratification not applied (regression target — continuous variable).
   Test set is held out completely until final evaluation.
""")


---
## 🔧 5. Build Preprocessing Pipeline

In [ ]:
# ── Numerical Pipeline ──
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # handle any residual NaNs
    ('scaler',  StandardScaler())                    # zero mean, unit variance
])

# ── Categorical Pipeline ──
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ── Binary columns: pass through (no transform needed) ──
# ── Combine with ColumnTransformer ──
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline,   numerical_cols),
    ('cat', categorical_pipeline, categorical_cols),
    ('bin', 'passthrough',        binary_cols),
], remainder='drop')

print("✅ Preprocessing pipeline built")
print("\nPipeline structure:")
print("  ColumnTransformer")
print("    ├── num → [SimpleImputer(median)] → [StandardScaler]")
print("    ├── cat → [SimpleImputer(mode)]   → [OneHotEncoder]")
print("    └── bin → passthrough")


---
## ⚙️ 6. Fit on Train, Transform Train & Test

In [ ]:
# Fit ONLY on train data — prevents data leakage
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)   # transform only

print(f"✅ Pipeline fit and transform complete")
print(f"  X_train shape before: {X_train.shape}")
print(f"  X_train shape after : {X_train_processed.shape}")
print(f"  X_test  shape before: {X_test.shape}")
print(f"  X_test  shape after : {X_test_processed.shape}")

# Get final feature names after OHE
ohe_features = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_cols).tolist()
final_feature_names = numerical_cols + ohe_features + binary_cols
print(f"\nTotal features after encoding: {len(final_feature_names)}")

print("""
📌 Decision: fit_transform on train only.
   transform (not fit_transform) on test.
   This avoids data leakage — test stats must not influence scaling/encoding.
""")


---
## 🎯 7. Feature Selection — SelectKBest (f_regression)

In [ ]:
# Select top K features based on F-statistic with target
from sklearn.feature_selection import SelectKBest, f_regression

selector = SelectKBest(score_func=f_regression, k='all')
selector.fit(X_train_processed, y_train)

scores = pd.DataFrame({
    'Feature': final_feature_names,
    'F_Score': selector.scores_,
    'P_Value': selector.pvalues_
}).sort_values('F_Score', ascending=False)

print("Top 20 features by F-score:")
print(scores.head(20).to_string(index=False))

# Plot top 20
fig, ax = plt.subplots(figsize=(10, 7))
top20 = scores.head(20)
colors = ['#27AE60' if p < 0.05 else '#E74C3C' for p in top20['P_Value']]
bars = ax.barh(top20['Feature'][::-1], top20['F_Score'][::-1],
    color=colors[::-1], edgecolor='white', alpha=0.85)
ax.set_title('Top 20 Features — F-Regression Score\n(Green = statistically significant p<0.05)', fontweight='bold')
ax.set_xlabel('F-Score')
plt.tight_layout()
plt.savefig('pp_02_feature_selection.png', bbox_inches='tight')
plt.show()


In [ ]:
# Apply selection — keep top 20 significant features
K = 20
selector_k = SelectKBest(score_func=f_regression, k=K)
X_train_selected = selector_k.fit_transform(X_train_processed, y_train)
X_test_selected  = selector_k.transform(X_test_processed)

selected_mask = selector_k.get_support()
selected_features = [f for f, m in zip(final_feature_names, selected_mask) if m]

print(f"✅ Selected top {K} features:")
for f in selected_features:
    print(f"  • {f}")

print(f"\nShape after selection:")
print(f"  X_train: {X_train_selected.shape}")
print(f"  X_test : {X_test_selected.shape}")

print("""
📌 Decision: SelectKBest with f_regression keeps the top 20 features
   most correlated with log_price. Reduces dimensionality after OHE expansion,
   removes noisy/redundant features, speeds up model training.
""")


---
## 🔒 8. Data Leakage Check

In [ ]:
print("Data Leakage Verification")
print("-" * 40)
print(f"  Preprocessor fitted on : TRAIN only ✅")
print(f"  Test transformed with  : Train stats ✅")
print(f"  Target in X_train      : {'log_price' in X_train.columns} ✅")
print(f"  Price_euros in X_train : {'Price_euros' in X_train.columns} ✅")
print(f"  Train/Test overlap     : {len(set(X_train.index) & set(X_test.index))} rows ✅")
print("-" * 40)
print("✅ No data leakage detected")


---
## 💾 9. Save Preprocessor & Processed Data

In [ ]:
import os
os.makedirs('models', exist_ok=True)

# Save preprocessor pipeline
joblib.dump(preprocessor, 'models/preprocessor.pkl')
joblib.dump(selector_k,   'models/feature_selector.pkl')

# Save processed arrays for model training
np.save('X_train_processed.npy', X_train_selected)
np.save('X_test_processed.npy',  X_test_selected)
np.save('y_train.npy', y_train.values)
np.save('y_test.npy',  y_test.values)

# Save selected feature names
pd.Series(selected_features).to_csv('selected_features.csv', index=False)

# Save train/test indices for reference
X_train.to_csv('X_train_raw.csv', index=True)
X_test.to_csv('X_test_raw.csv',   index=True)

print("✅ Saved:")
print("  models/preprocessor.pkl")
print("  models/feature_selector.pkl")
print("  X_train_processed.npy")
print("  X_test_processed.npy")
print("  y_train.npy / y_test.npy")
print("  selected_features.csv")
print("  X_train_raw.csv / X_test_raw.csv")


---
## ✅ 10. Preprocessing Pipeline Summary

In [ ]:
print("=" * 60)
print("      PREPROCESSING PIPELINE SUMMARY")
print("=" * 60)

steps = [
    ("Train-Test Split",    "80% train / 20% test / random_state=42"),
    ("Numerical cols",      "SimpleImputer(median) → StandardScaler"),
    ("Categorical cols",    "SimpleImputer(mode) → OneHotEncoder"),
    ("Binary cols",         "Passthrough (no transform)"),
    ("Fit rule",            "fit_transform on train / transform on test"),
    ("Feature Selection",   f"SelectKBest f_regression → top {K} features"),
    ("Leakage check",       "Verified — no leakage"),
    ("Saved artifacts",     "preprocessor.pkl, feature_selector.pkl"),
]

for step, detail in steps:
    print(f"  ✅ {step:<22}: {detail}")

print("=" * 60)
print(f"  Final X_train shape : {X_train_selected.shape}")
print(f"  Final X_test  shape : {X_test_selected.shape}")
print("=" * 60)
print("\n📌 Next Step → Model Training (Step 6)")

# Auto-download key files
from google.colab import files
for fname in ['models/preprocessor.pkl', 'models/feature_selector.pkl',
              'selected_features.csv', 'laptop_price_featured.csv']:
    try:
        files.download(fname)
    except:
        pass
print("\n✅ Key files downloaded!")
